In [1]:
from scipy.stats import poisson
import numpy as np
import requests
import os
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime

load_dotenv()

API_KEY = os.getenv("SPORTS_API_KEY")
BASE_URL = "https://v3.football.api-sports.io"
Headers = {'x-apisports-key': API_KEY}
Premier_League = 39
Season = 2023
ALLOWED_SEASONS = [2022, 2023, 2024]
AS_OF_DATE = datetime(2024, 1, 1)


In [ ]:
response = requests.get("https://v3.football.api-sports.io/leagues", headers=headers)
data = response.json()

# peek at structure first, don't dump everything
print(data.keys())
print(json.dumps(data['response'][0], indent=2))  # just the first league

In [5]:
print(data['errors'])
print(data['results'])


{'token': 'Missing application key, Check our documentation on how to add your API key in headers.', 'error': '4xHe'}
0


In [9]:
load_dotenv()

False

In [7]:
from pathlib import Path
print(Path.cwd())

C:\Users\marca\Football-Analytics\data


In [19]:
for league in data['response']:
    if "Champions League" in league['league']['name']:
        print(league['league']['id'], league['league']['name'], league['country']['name'])

2 UEFA Champions League World
27 OFC Champions League World
16 CONCACAF Champions League World
17 AFC Champions League Elite World
18 AFC Champions League Two World
525 UEFA Champions League Women World
12 CAF Champions League World
823 Nasjonal U19 Champions League Norway
1140 AFC Women's Champions League World
1162 AGCFF Gulf Champions League World
1164 CAF Women's Champions League World


Champons league id = 2

In [24]:
CURRENT_SEASON = 2024  # API-Football uses the year the season started

response = requests.get(
    "https://v3.football.api-sports.io/teams",
    headers=headers,
    params={"league": 2, "season": CURRENT_SEASON}
)
data = response.json()
print(data['errors'])
print(data['results'])
print(data['response'][3])  # peek at one team's shape

[]
81
{'team': {'id': 66, 'name': 'Aston Villa', 'code': 'AST', 'country': 'England', 'founded': 1874, 'national': False, 'logo': 'https://media.api-sports.io/football/teams/66.png'}, 'venue': {'id': 495, 'name': 'Villa Park', 'address': 'Trinity Road', 'city': 'Birmingham', 'capacity': 42824, 'surface': 'grass', 'image': 'https://media.api-sports.io/football/venues/495.png'}}


In [17]:
response = requests.get(
    "https://v3.football.api-sports.io/fixtures/headtohead",
    headers=headers,
    params={"h2h": f"{40}-{66}"}
)
data = response.json()
print(data['results'])
print(data['response'][0])

31
{'fixture': {'id': 157338, 'referee': 'Paul Tierney, England', 'timezone': 'UTC', 'date': '2020-07-05T15:30:00+00:00', 'timestamp': 1593963000, 'periods': {'first': 1593963000, 'second': 1593966600}, 'venue': {'id': 550, 'name': 'Anfield', 'city': 'Liverpool'}, 'status': {'long': 'Match Finished', 'short': 'FT', 'elapsed': 90, 'extra': None}}, 'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2019, 'round': 'Regular Season - 33', 'standings': True}, 'teams': {'home': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png', 'winner': True}, 'away': {'id': 66, 'name': 'Aston Villa', 'logo': 'https://media.api-sports.io/football/teams/66.png', 'winner': False}}, 'goals': {'home': 2, 'away': 0}, 'score': {'halftime': {'home': 0, 'away': 0}, 'fulltime': {'home': 2, 'away': 0}, 'extratime': {'home': None, 'a

In [8]:
def get_injuries(team_id, season):
    response = requests.get("https://v3.football.api-sports.io/injuries", headers=headers, params={"team": team_id, "season": season})
    return response.json()

In [13]:
def get_head_to_head(team_a_id, team_b_id, season):
    response = requests.get(f"{BASE_URL}/fixtures/headtohead", headers=Headers,
                             params={"h2h": f"{team_a_id}-{team_b_id}",
                                     "league": Champions_Leagues_ID, "season": season})
    return response.json()

In [12]:
def summarize_h2h(data, team_a_id, team_b_id):
    fixutres = data["response"]
    team_a_wins = 0
    team_b_wins = 0
    draws = 0
    for f in fixutres:
        home_id = f["teams"]["home"]["id"]
        home_win = f["teams"]["home"]["winner"]
        away_win = f["teams"]["away"]["winner"]
        if home_win:
            team_a_wins +=1 if home_id == team_a_id else 0
            team_b_wins +=1 if home_id == team_b_id else 0
        elif away_win:
            team_a_wins +=1 if home_id == team_b_id else 0
            team_b_wins +=1 if home_id == team_a_id else 0
        else:
            draws +=1

    return {
        "total_matches": len(fixutres),
        "team_a_wins": team_a_wins,
        "team_b_wins": team_b_wins,
        "draws": draws
    }

In [18]:
summarize_h2h(data, 40, 66)

{'total_matches': 31, 'team_a_wins': 17, 'team_b_wins': 7, 'draws': 7}

In [7]:
def get_squad(team_id):
    response = requests.get(f"{BASE_URL}/players/squads", headers=Headers,
                             params={"team": team_id,})
    return response.json()

In [8]:
get_squad(40)

{'get': 'players/squads',
 'parameters': {'team': '40'},
 'errors': [],
 'results': 1,
 'paging': {'current': 1, 'total': 1},
 'response': [{'team': {'id': 40,
    'name': 'Liverpool',
    'logo': 'https://media.api-sports.io/football/teams/40.png'},
   'players': [{'id': 280,
     'name': 'Alisson Becker',
     'age': 33,
     'number': 1,
     'position': 'Goalkeeper',
     'photo': 'https://media.api-sports.io/football/players/280.png'},
    {'id': 24760,
     'name': 'G. Mamardashvili',
     'age': 25,
     'number': 25,
     'position': 'Goalkeeper',
     'photo': 'https://media.api-sports.io/football/players/24760.png'},
    {'id': 415992,
     'name': 'K. Miściur',
     'age': 18,
     'number': 13,
     'position': 'Goalkeeper',
     'photo': 'https://media.api-sports.io/football/players/415992.png'},
    {'id': 342467,
     'name': 'Á. Pécsi',
     'age': 20,
     'number': 41,
     'position': 'Goalkeeper',
     'photo': 'https://media.api-sports.io/football/players/342467.pn

In [9]:
def get_injuries(team_id, season):
    response = requests.get(f"{BASE_URL}/injuries", headers=Headers, params={"team": team_id, "season": season})
    return response.json()

In [ ]:
get_injuries(40, 2024)

In [14]:
for season in [2021, 2022, 2023, 2024,2025]:
    response = requests.get(f"{BASE_URL}/teams", headers=Headers,
                             params={"league": 39, "season": season})
    print(season, response.json()['results'])

2021 0
2022 20
2023 20
2024 20
2025 0


In [2]:
def get_recent_form(team_id, last = 5):
    response = requests.get(f"{BASE_URL}/fixtures", headers=Headers, params={"team": team_id, "last": last})
    return response.json()

In [3]:
get_recent_form(40)

{'get': 'fixtures',
 'parameters': {'team': '40', 'last': '5'},
 'errors': {'plan': 'Free plans do not have access to the Last parameter.'},
 'results': 0,
 'paging': {'current': 1, 'total': 1},
 'response': []}

In [2]:
def get_fixtures(league, season):
    response = requests.get(f"{BASE_URL}/fixtures", headers = Headers, params={"league": league, "season": season})
    return response.json()

In [5]:
get_fixtures(39, 2024)

SyntaxError: invalid syntax (480756170.py, line 1)

In [8]:
# does squads endpoint accept season at all?
response = requests.get(f"{BASE_URL}/players/squads", headers= Headers,
                         params={"team": 40, "season": 2023})
print(response.json()['errors'])  # if it complains about unexpected param, confirms it's ignored

{'season': 'The Season field do not exist.'}


In [9]:
# compare against the players endpoint
response2 = requests.get(f"{BASE_URL}/players", headers= Headers,
                          params={"team": 40, "season": 2023, "league": 39})
print(response2.json()['results'])

20


In [12]:
def get_squad(team_id, season= 2023, league_id= 39):
    response = requests.get(
        f"{BASE_URL}/players",
        headers= Headers,
        params={"team": team_id, "season": season, "league": league_id}
    )
    return response.json()

In [16]:
raw = get_squad(40, season=2023)
print(raw['response'][17])

{'player': {'id': 284, 'name': 'J. Gomez', 'firstname': 'Joseph Dave', 'lastname': 'Gomez', 'age': 28, 'birth': {'date': '1997-05-23', 'place': 'London', 'country': 'England'}, 'nationality': 'England', 'height': '188', 'weight': '80', 'injured': False, 'photo': 'https://media.api-sports.io/football/players/284.png'}, 'statistics': [{'team': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png'}, 'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2023}, 'games': {'appearences': 32, 'lineups': 17, 'minutes': 1777, 'number': None, 'position': 'Defender', 'rating': '6.935483', 'captain': False}, 'substitutes': {'in': 15, 'out': 5, 'bench': 19}, 'shots': {'total': 14, 'on': 2}, 'goals': {'total': 0, 'conceded': 0, 'assists': 1, 'saves': None}, 'passes': {'total': 1148, 'key': 22, 'accuracy': None}, 'tackles'

In [17]:
print(raw['paging']) 

{'current': 1, 'total': 3}


In [20]:
def get_injuries(team_id, season= Season, as_of_date=AS_OF_DATE):
    response = requests.get(
        f"{BASE_URL}/injuries",
        headers= Headers,
        params={"team": team_id, "season": season}
    )
    data = response.json()
    injuries = data['response']

    if as_of_date is not None:
        cutoff = as_of_date.timestamp()
        injuries = [i for i in injuries if i['fixture']['timestamp'] < cutoff]

    return injuries

In [22]:
raw = get_injuries(40)
print(raw[0] if raw else "no injuries found")

{'player': {'id': 310187, 'name': 'S. Bajcetic', 'photo': 'https://media.api-sports.io/football/players/310187.png', 'type': 'Questionable', 'reason': 'Muscle Injury'}, 'team': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png'}, 'fixture': {'id': 1035045, 'timezone': 'UTC', 'date': '2023-08-13T15:30:00+00:00', 'timestamp': 1691940600}, 'league': {'id': 39, 'season': 2023, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg'}}


In [38]:
def get_head_to_head(team_a_id, team_b_id, league_id= Premier_League, as_of_date = AS_OF_DATE):
    response = requests.get(
        f"{BASE_URL}/fixtures/headtohead",
        headers= Headers,
        params={"h2h": f"{team_a_id}-{team_b_id}", "league": league_id, "season": 2024 }
    )
    data = response.json()
    fixtures = data['response']

    if as_of_date is not None:
        cutoff = int(as_of_date.timestamp())
        fixtures = [f for f in fixtures if f['fixture']['timestamp'] < cutoff]

    return fixtures

In [39]:
get_head_to_head(40, 66)

[{'fixture': {'id': 1208127,
   'referee': 'D. Coote',
   'timezone': 'UTC',
   'date': '2024-11-09T20:00:00+00:00',
   'timestamp': 1731182400,
   'periods': {'first': 1731182400, 'second': 1731186000},
   'venue': {'id': 550, 'name': 'Anfield', 'city': 'Liverpool'},
   'status': {'long': 'Match Finished',
    'short': 'FT',
    'elapsed': 90,
    'extra': 4}},
  'league': {'id': 39,
   'name': 'Premier League',
   'country': 'England',
   'logo': 'https://media.api-sports.io/football/leagues/39.png',
   'flag': 'https://media.api-sports.io/flags/gb-eng.svg',
   'season': 2024,
   'round': 'Regular Season - 11',
   'standings': True},
  'teams': {'home': {'id': 40,
    'name': 'Liverpool',
    'logo': 'https://media.api-sports.io/football/teams/40.png',
    'winner': True},
   'away': {'id': 66,
    'name': 'Aston Villa',
    'logo': 'https://media.api-sports.io/football/teams/66.png',
    'winner': False}},
  'goals': {'home': 2, 'away': 0},
  'score': {'halftime': {'home': 1, 'away'

In [18]:
response = requests.get(
        f"{BASE_URL}/fixtures/headtohead",
        headers= Headers,
        params={"h2h": f"{40}-{66}", "league": 39, "season": 2023}
    )

In [19]:
print(response.json())

{'get': 'fixtures/headtohead', 'parameters': {'h2h': '40-66', 'league': '39', 'season': '2023', 'date': '2024-01-01 00:00:00'}, 'errors': {'date': 'The Date field must contain a valid date: Y-m-d.'}, 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': []}


In [13]:
int(AS_OF_DATE.timestamp())

1704063600

In [5]:
def get_all_league_fixtures(season, league_id=Premier_League):
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers=Headers,
        params={"league": league_id, "season": season}
    )
    data = response.json()
    return [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

all_fixtures_2023 = get_all_league_fixtures(2023)
print(len(all_fixtures_2023))

380


In [3]:
def get_matchday_fixtures(round_name, season=Season, league_id=Premier_League):
    """
    Get all fixtures for a specific matchday/round, e.g. 'Regular Season - 21'.
    Returns finished fixtures only, with real scores included for comparison.
    """
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers=Headers,
        params={"league": league_id, "season": season, "round": round_name}
    )
    data = response.json()
    return [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

In [6]:
cutoff = int(AS_OF_DATE.timestamp())
upcoming = [f for f in all_fixtures_2023 if f['fixture']['timestamp'] >= cutoff]
upcoming.sort(key=lambda f: f['fixture']['timestamp'])

# peek at the round name of the very next matches after the cutoff
print(upcoming[0]['league']['round'])
print(upcoming[0]['fixture']['date'])

Regular Season - 20
2024-01-01T20:00:00+00:00


In [7]:
next_matchday = get_matchday_fixtures("Regular Season - 21")  # use whatever round name you found
print(len(next_matchday))
for f in next_matchday:
    print(f['teams']['home']['name'], f['goals']['home'], '-', f['goals']['away'], f['teams']['away']['name'])

10
Burnley 1 - 1 Luton
Chelsea 1 - 0 Fulham
Newcastle 2 - 3 Manchester City
Everton 0 - 0 Aston Villa
Manchester United 2 - 2 Tottenham
Arsenal 5 - 0 Crystal Palace
Brentford 3 - 2 Nottingham Forest
Sheffield Utd 2 - 2 West Ham
Bournemouth 0 - 4 Liverpool
Brighton 0 - 0 Wolves


NameError: name 'get_teams' is not defined

In [9]:
for name, tid in team_ids.items():
    print(name, tid, len(fixtures_cache[tid]))

NameError: name 'team_ids' is not defined

In [2]:

from pathlib import Path
import os

# Check sizes to confirm which is real vs empty
print("Current dir file size:", Path("football.db").stat().st_size)
print("Parent dir file size:", Path("../football.db").stat().st_size)

Current dir file size: 0
Parent dir file size: 917504


In [4]:

import sqlite3
conn = sqlite3.connect("../football.db")
cursor = conn.execute("""
    SELECT team_id, name, COUNT(DISTINCT league_id) as league_count
    FROM teams
    GROUP BY team_id, name
    HAVING league_count > 1
""")
results = cursor.fetchall()
conn.close()
print(f"Teams associated with more than one league: {len(results)}")
for r in results[:20]:
    print(r)

Teams associated with more than one league: 34
(33, 'Manchester United', 2)
(34, 'Newcastle', 2)
(40, 'Liverpool', 2)
(42, 'Arsenal', 2)
(47, 'Tottenham', 2)
(49, 'Chelsea', 2)
(50, 'Manchester City', 2)
(66, 'Aston Villa', 2)
(79, 'Lille', 2)
(81, 'Marseille', 2)
(85, 'Paris Saint Germain', 2)
(91, 'Monaco', 2)
(106, 'Stade Brestois 29', 2)
(116, 'Lens', 2)
(157, 'Bayern München', 2)
(165, 'Borussia Dortmund', 2)
(168, 'Bayer Leverkusen', 2)
(169, 'Eintracht Frankfurt', 2)
(172, 'VfB Stuttgart', 2)
(173, 'RB Leipzig', 2)


In [5]:
type(results)

list